# Tutorial 1 — Photometry: VHS 1256 b

**What you'll learn:** How to run a photometric atmosphere retrieval with ForMoSA v2.0 —
loading multi-instrument broadband photometry, configuring the analysis with Python dataclasses,
adapting a model grid, running nested sampling, and interpreting the results.

**Target:** VHS J125601.92−125723.9 b (VHS 1256 b) — a young (~140 Myr), planetary-mass companion
at ~22 pc with one of the most extreme red colours of any directly-imaged companion known.
It sits on the L/T transition and hosts a spectrum dominated by thick, silicate clouds.
It has been observed by SPHERE and NACO at the VLT, and more recently by JWST/NIRCam and JWST/MIRI.

**Data:** 14 broadband photometric points spanning 1–16 µm (SPHERE + NACO + JWST NIRCam + JWST MIRI).

**Grid:** BT-Settl (Allard et al. 2012) — Teff: 1000–3000 K, log g: 2.5–5.5

**Estimated runtime:**
- Grid adaptation: < 15 s
- Nested sampling (100 live points, nestle): ~1 min

**References:**
Miles et al. (2023, ApJL, 946, L6); Petrus et al. (2023, A&A, 670, L9)


## Section 0: Setup

Run these four cells before anything else. They check your environment, create the
working directories, download the observation data, and download the model grid.


In [ ]:
# Cell A: Environment check
import sys

try:
    import ForMoSA
    print(f"ForMoSA {ForMoSA.__version__} — OK")
except ImportError:
    raise ImportError(
        "ForMoSA is not installed.\n"
        "Run:  pip install ForMoSA && conda install dask netCDF4 bottleneck"
    )

print(f"Python {sys.version.split()[0]}")


In [ ]:
# Cell B: Workspace setup
from pathlib import Path

TUTORIAL_DIR = Path(".").resolve()   # directory containing this notebook

for d in ["data", "adapted_grid", "results", "grid"]:
    (TUTORIAL_DIR / d).mkdir(exist_ok=True)

print(f"Working directory : {TUTORIAL_DIR}")
print("Subdirectories    : data/  adapted_grid/  results/  grid/")


In [ ]:
# Cell C: Data download + validation
import urllib.request
from astropy.io import fits

DATA_FILE = TUTORIAL_DIR / "data" / "VHS1256b_photometry.fits"
DATA_URL  = (
    "https://github.com/exoAtmospheres/ForMoSA/releases/download/"
    "tutorial-data-v1/VHS1256b_photometry.fits"
)

if not DATA_FILE.exists():
    print("Downloading observation data (~1 MB)...")
    urllib.request.urlretrieve(DATA_URL, DATA_FILE)
    print("Done.")
else:
    print(f"Data already present: {DATA_FILE.name}")

# Validate FITS extensions
REQUIRED = {"WAV", "WAVE_UNIT", "FLX", "ERR", "FAC", "INS", "FILT"}
with fits.open(DATA_FILE) as hdul:
    found = {ext.name for ext in hdul[1:]}   # skip PRIMARY

missing = REQUIRED - {k.upper() for k in found}
if missing:
    raise RuntimeError(f"Missing FITS extensions: {missing}")

print("\nFITS extensions (required marked ✓):")
for name in sorted(found):
    mark = "✓" if name.upper() in REQUIRED else " "
    print(f"  {mark} {name}")
print("\nAll required extensions present — data is valid.")


In [ ]:
# Cell D: Grid download
# The full BT-Settl grid is ~1 GB. Once downloaded, keep it —
# it works for all ForMoSA tutorials and your own science.
import urllib.request

GRID_FILE = TUTORIAL_DIR / "grid" / "BT-Settl.nc"
GRID_URL  = (
    "https://github.com/exoAtmospheres/ForMoSA/releases/download/"
    "tutorial-data-v1/BT-Settl.nc"
)

# Override GRID_FILE here if you already have the grid elsewhere:
# GRID_FILE = Path("/path/to/your/BT-Settl.nc")

if not GRID_FILE.exists():
    print("Downloading BT-Settl model grid (~1 GB). This takes a few minutes.\n")
    try:
        from tqdm import tqdm
        class _Progress(tqdm):
            def update_to(self, b=1, bs=1, ts=None):
                if ts: self.total = ts
                self.update(b * bs - self.n)
        with _Progress(unit="B", unit_scale=True, desc="BT-Settl.nc") as t:
            urllib.request.urlretrieve(GRID_URL, GRID_FILE, reporthook=t.update_to)
    except ImportError:
        def _progress(count, bs, total):
            pct = min(100, count * bs / total * 100)
            print(f"\r  {pct:.1f}%  {count*bs/1e6:.1f}/{total/1e6:.1f} MB",
                  end="", flush=True)
        urllib.request.urlretrieve(GRID_URL, GRID_FILE, reporthook=_progress)
        print()
    print(f"\nSaved: {GRID_FILE}")
else:
    print(f"Grid already present: {GRID_FILE.name}")

import xarray as xr
ds = xr.open_dataset(GRID_FILE, decode_cf=False)
par_names = ds.attrs.get("par", [])
par_units = ds.attrs.get("unit", [])
print(f"\nGrid dimensions: {dict(ds.sizes)}")
for i, (key, name, unit) in enumerate(zip(["par1", "par2", "par3", "par4"],
                                           par_names or ["par1","par2","par3","par4"],
                                           par_units or ["","","",""])):
    if key in ds.coords:
        vals = ds[key].values
        print(f"  {name} {unit}: {vals[0]:.1f} → {vals[-1]:.1f}  ({len(vals)} points)")


## Section 1: The science

### Why photometry?

A spectrum tells you the detailed shape of an atmosphere's emission as a function of wavelength.
Photometry gives you the *integrated* flux in broad bands — less information per data point, but
easier to obtain across a wide wavelength range and across many instruments.

From photometry alone you can constrain:
- **Teff** (effective temperature) — sets the overall luminosity and colour
- **log g** (surface gravity) — affects the pressure-broadening of molecular bands and the
  cloud sedimentation efficiency
- **Radius** — via the Stefan-Boltzmann law once Teff and luminosity are known
- **Bolometric luminosity** — integrating the SED over all bands

### VHS 1256 b

VHS 1256 b was first identified by Gauza et al. (2015) as an exceptionally red, young companion
to the M dwarf binary VHS J1256−1257. Its youth (~140 Myr, Upper-Centaurus Lupus association)
means low surface gravity and active cloud formation — exactly why it sits on the L/T transition
and looks so red. JWST observations (Miles et al. 2023) confirmed silicate absorption at 8–10 µm
and CO₂ at 4.2 µm, making it one of the most well-characterised sub-stellar atmospheres to date.

We fit 14 photometric points spanning 1.2–15.5 µm:
- **SPHERE/IRDIS**: H2, H3, K1, K2 bands
- **NACO**: L', NB4.05, M' bands
- **JWST/NIRCam**: F250M, F300M, F356W, F410M, F444W
- **JWST/MIRI**: F1140C, F1550C

### Expected results

Literature retrieval (Petrus et al. 2023): Teff ≈ 1200–1400 K, log g ≈ 3.5–4.5.
BT-Settl struggles with the very red colours of VHS 1256 b (no cloud microphysics),
so we expect the fit to be imperfect — but it's a great benchmark case.


## Section 2: Inspect the data

In [ ]:
from astropy.io import fits
import matplotlib.pyplot as plt
import numpy as np

with fits.open(DATA_FILE) as hdul:
    wav  = hdul["WAV"].data.astype(float)       # wavelength (µm)
    flx  = hdul["FLX"].data.astype(float)       # flux (W/m²/µm)
    err  = hdul["ERR"].data.astype(float)       # 1-σ uncertainty
    fac  = hdul["FAC"].data                     # facility names
    ins  = hdul["INS"].data                     # instrument names
    filt = hdul["FILT"].data                    # filter names

print(f"Number of photometric points: {len(wav)}")
print(f"Wavelength range: {wav.min():.2f} – {wav.max():.2f} µm\n")
print(f"{'Filter':<20} {'λ (µm)':>8} {'Flux':>14} {'SNR':>6}")
print("-" * 52)
for w, f, e, fi in sorted(zip(wav, flx, err, filt), key=lambda x: x[0]):
    print(f"{fi:<20} {w:>8.3f} {f:>14.3e} {f/e:>6.1f}")


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.errorbar(wav, flx, yerr=err, fmt="o", color="#2E86AB", ecolor="gray",
            capsize=4, label="VHS 1256 b photometry")
ax.set_xlabel(r"Wavelength ($\mu$m)")
ax.set_ylabel(r"Flux (W m$^{-2}$ $\mu$m$^{-1}$)")
ax.set_title("VHS 1256 b — Spectral Energy Distribution")
ax.set_xscale("log")
ax.set_yscale("log")
ax.legend()
plt.tight_layout()
plt.show()


## Section 3: Configure the analysis

ForMoSA v2.0 uses four Python dataclasses to configure a run. Each dataclass
groups related settings — you only need to set the ones that differ from the defaults.

| Dataclass | Controls |
|-----------|----------|
| `ConfigPath` | File paths (observation, grid, output directories) |
| `ConfigAdapt` | How the grid is resampled to match the observations |
| `ConfigInversion` | Wavelength fitting window, nested sampling algorithm |
| `ConfigParameters` | Prior on each free parameter |


In [ ]:
from ForMoSA.config.global_config import ConfigPath, ConfigAdapt, ConfigInversion, ConfigParameters

# ── Paths ──
config_path = ConfigPath(
    observation_path=[str(DATA_FILE)],          # list of FITS files
    adapt_store_path=str(TUTORIAL_DIR / "adapted_grid"),
    result_path=str(TUTORIAL_DIR / "results"),
    model_path=str(GRID_FILE),
)

# ── Adaptation ──
# Defaults are fine for photometry: no continuum removal (res_cont="NA"),
# target resolution = observation resolution (target_res_obs="obs").
config_adapt = ConfigAdapt()

# ── Inversion ──
config_inversion = ConfigInversion(
    wav_fit=["0.9, 16.0"],   # µm range covering all 14 filters
    ns_algo="nestle",        # use nestle in Jupyter (PyMultiNest needs MPI)
    npoints=100,             # live points (increase for production runs)
    logL_type=["chi2"],      # standard χ² likelihood
)

# ── Parameters ──
# Syntax for priors: ["type", "min", "max"] or ["constant", "value"]
# par1 = Teff (K), par2 = log g (dex) — read from grid attributes above
config_params = ConfigParameters(
    par1=["uniform", "1000", "2800"],  # Teff range from BT-Settl grid
    par2=["uniform", "3.0", "5.5"],   # log g range (young objects: low g)
    r=["uniform", "0.5", "2.0"],      # radius in R_Jupiter
    d=["constant", "22.0"],           # distance in pc (Gaia: 22.0 ± 0.2 pc)
    alpha=["uniform", "0.1", "10.0"], # analytical flux scaling factor
    # alpha accounts for uncertain absolute flux calibration across instruments.
    # Physical interpretation: alpha = 1 means the model matches flux perfectly.
)

print("Configuration ready.")
print(f"  Free parameters: par1 (Teff), par2 (log g), r, alpha")
print(f"  Fixed:           d = 22.0 pc")
print(f"  Wavelength fit:  {config_inversion.wav_fit[0]} µm")


## Section 4: Adapt the grid

The grid is pre-computed at native model resolution. Before fitting, ForMoSA
*adapts* it: it evaluates the model flux through each photometric filter,
producing a much smaller grid of synthetic photometry that is directly comparable
to your observations.

For photometry, adaptation is fast — there is no spectral convolution, just
filter integration via the SVO Filter Profile Service (downloaded automatically
the first time each filter is used).

Set `adapted=True` on subsequent runs to skip this step and load the cached grid.


In [ ]:
from ForMoSA import Analysis

# adapted=False: run adaptation and save to adapted_grid/
# adapted=True:  skip adaptation and load from adapted_grid/ (re-runs only)
adapted = False

analysis = Analysis(config_path, adapted=adapted, fitted=False)

if not adapted:
    print("Adapting grid to observations...")
    analysis.adapt(config_adapt, config_inversion)
    print("Adaptation complete. Set adapted=True on next run to skip this step.")


## Section 5: Run the nested sampling fit

Nested sampling explores the prior volume and computes the Bayesian evidence
(log Z) alongside the posterior distributions. We use **Nestle** here because
it runs in a single process — ideal for a notebook environment.

For production runs with more live points or more free parameters, switch to
`ns_algo="pymultinest"` and run with MPI (see Tutorial 6).

The `npoints=100` setting is a good starting point for a quick check.
For publication-quality posteriors, use 300–500 live points.


In [ ]:
from ForMoSA.config.global_config import Config_NS

config_ns = Config_NS()   # bundle of backend configs; defaults are fine

print("Running nested sampling...")
print(f"  Algorithm : {config_inversion.ns_algo}")
print(f"  Live pts  : {config_inversion.npoints}")
print(f"  Free params: par1, par2, r, alpha  (d is fixed)")
print()

analysis.nested_sampling(config_params, config_adapt, config_inversion, config_NS=config_ns)

print("\nFit complete. Results saved to results/")


## Section 6: Results

ForMoSA produces four diagnostic plots saved to `results/`:

| File | Shows |
|------|-------|
| `corner.pdf` | Posterior distributions and pairwise correlations |
| `chains.pdf` | Sample chains and weights (convergence check) |
| `radar.pdf` | Radar diagram of median ± 1σ for each parameter |
| `best_fit.pdf` | Best-fit model vs. observed data + residuals |


In [ ]:
# Plot all results
# The figures are also saved as PDFs in results/
analysis.plot(analysis.ns.results, plot_native_model=False)

# Print a numerical summary
print(analysis.ns.results.summary(sigma=1))


## Section 7: INI file alternative

The dataclass approach above is recommended — everything lives in one place
and is easy to version-control. If you prefer an INI file (closer to ForMoSA v1.x
and convenient for cluster runs), use `ConfigGenerator` to create a default
template and `ConfigLoader` to load it.


In [ ]:
from ForMoSA.config.global_config import ConfigGenerator, ConfigLoader, Config_NS

# 1. Generate a default config.ini in TUTORIAL_DIR
generator = ConfigGenerator()
generator.save(str(TUTORIAL_DIR), "config.ini")
print(f"Default config written to: {TUTORIAL_DIR / 'config.ini'}")
print("Open it in a text editor, fill in the paths and parameters, then load it:")


In [ ]:
# 2. Load the config (after editing the .ini manually or programmatically)
# cfg = ConfigLoader(str(TUTORIAL_DIR / "config.ini"))
# sections = cfg.load()
#
# 3. Run — identical to the dataclass workflow:
# analysis = Analysis(cfg.config["config_path"], adapted=False, fitted=False)
# analysis.adapt(cfg.config["config_adapt"], cfg.config["config_inversion"])
# config_ns = Config_NS(
#     nestle=cfg.config["config_nestle"],
#     pymultinest=cfg.config["config_pymultinest"],
#     ultranest=cfg.config["config_ultranest"],
# )
# analysis.nested_sampling(
#     cfg.config["config_parameters"],
#     cfg.config["config_adapt"],
#     cfg.config["config_inversion"],
#     config_NS=config_ns,
# )
# analysis.plot(analysis.ns.results)
print("See the comments above for the full INI-based workflow.")


## Section 8: Next steps

- **Tutorial 2 — Spectroscopy (AB Pic b):** Same adapt → sample → plot loop,
  but with a K-band spectrum. Introduces resolution adaptation and radial velocity.
- **Getting started guide:** `docs/getting_started/config_file.md` for a full
  parameter reference (prior types, MOSAIC indexing, HCHR options).
- **API reference:** `docs/api/` for the full class and method documentation.

To fit your own target, copy this notebook and change:
1. `DATA_FILE` → your `.fits` observation file
2. `GRID_FILE` → your model grid (`.nc`)
3. `config_params` → priors appropriate for your target
